# SatPM V6.GL.03 — Monthly PM₂.₅ × Bolivia (339 historical municipalities)

Adapted from the supplied Peru workflow. Scientific processing of South America monthly NetCDF rasters is retained. This version locks the **historical 339-municipality geography** and publishes validated outputs to the public Hugging Face dataset [`faviolc/gdo4Bolivia`](https://huggingface.co/datasets/faviolc/gdo4Bolivia).

**Author:** Favio Leiva Cárdenas  
**Geography:** 339 historical Bolivian municipalities (not the new 343-unit layer)  
**Period:** January 1998–December 2024; 324 months; expected **109,836** municipality-month rows.  
**Source map:** `quarcs-lab/gdo4bolivia/raw/boundaries/bolivia339geoqueryOpt.geojson`.  
**Crosswalk:** `raw/lookups/municipalities339_gadm.csv`.  
**Important:** Data outputs are public. Never print/upload credentials or sensitive notebook outputs.

## Step 0. Install dependencies

Run this cell in Google Colab before the rest of the notebook. The workflow uses Hugging Face Hub for downloads, GeoPandas for municipality polygons, Xarray/Rioxarray for NetCDF handling, Rasterio for temporary GeoTIFF writing, and Rasterstats for zonal statistics.

In [ ]:
!pip -q install huggingface_hub
!pip -q install geopandas shapely pyogrio fiona pyproj rtree
!pip -q install rasterio rasterstats rioxarray xarray netCDF4 h5netcdf
!pip -q install pandas numpy pyarrow fastparquet tqdm joblib matplotlib
!pip -q install joblib tqdm requests nbformat

## Step 1. Imports and runtime settings

In [ ]:
# ------------------------------------------------------------------------------
# Thread control
# ------------------------------------------------------------------------------
# Keep numerical libraries from oversubscribing Colab CPU threads while joblib
# handles month-level parallelism.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

# ------------------------------------------------------------------------------
# Standard library
# ------------------------------------------------------------------------------
import re
import json
import time
import shutil
import traceback
from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------------------------
# Scientific stack
# ------------------------------------------------------------------------------
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray  # noqa: F401. Required to activate the .rio accessor.
import rasterio
from rasterio.errors import NotGeoreferencedWarning
from rasterstats import zonal_stats
from shapely.geometry import box
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# ------------------------------------------------------------------------------
# Hugging Face
# ------------------------------------------------------------------------------
from huggingface_hub import HfApi, hf_hub_download, login, upload_folder

try:
    from google.colab import userdata
    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 160)

print("Imports completed.")
print("Current UTC time:", datetime.now(timezone.utc).isoformat())
print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)
print("xarray:", xr.__version__)
print("rasterio:", rasterio.__version__)
print("Running in Colab:", IN_COLAB)

## Step 2. Configuration

Historical 339-unit source URLs below are intentionally fixed to the original `raw/` map, **not** `geo/boundaries/municipality.parquet` (343 units). Run the smoke test and the single-year check before enabling full production.

In [ ]:
HF_TOKEN_SECRET_NAME = "HF_Writing"  # private Colab secret with dataset write permission
PM25_REPO_ID = "faviolc/monthlypm25"
PM25_REPO_TYPE = "dataset"
PM25_REGION_CODE = "SA"
PM25_BASE_FOLDER = "Monthly/SA"
PRODUCT_NAME = "SatPM V6.GL.03 Monthly PM2.5"
PM25_UNITS = "micrograms_per_cubic_meter"
PM25_RESOLUTION_LABEL = "0.01_degree"
START_YEAR, END_YEAR = 1998, 2024
MONTHS = list(range(1, 13))
EXPECTED_ADMIN_UNITS = 339

# Pin these two files to the *same* audited Git commit if reproducible boundaries are required.
GDO_GIT_REF = "main"  # change to a commit SHA after confirming the 339 files
GDO_RAW_BASE = f"https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/{GDO_GIT_REF}"
MAP_URL = GDO_RAW_BASE + "/raw/boundaries/bolivia339geoqueryOpt.geojson"
LOOKUP_URL = GDO_RAW_BASE + "/raw/lookups/municipalities339_gadm.csv"
MAP_REPO_ID = "quarcs-lab/gdo4bolivia"
MAP_HF_PATH = "raw/boundaries/bolivia339geoqueryOpt.geojson"  # provenance, GitHub path
MAP_REVISION = GDO_GIT_REF
LOCAL_ADMIN_FILE = None  # local GeoJSON alternative
LOCAL_LOOKUP_FILE = None  # local CSV alternative

BASE_STATS = ["count", "min", "max", "median", "mean", "sum", "std"]
ALL_TOUCHED = False
RASTER_NODATA = -9999.0
PM25_NEGATIVE_TO_NODATA = True
PM25_ZERO_IS_VALID = True
PROCESSING_MODE = "safe"
MODE_SETTINGS = {"safe": {"month_workers": 2, "parallel_backend": "loky"},
                 "balanced": {"month_workers": 4, "parallel_backend": "loky"},
                 "aggressive": {"month_workers": 6, "parallel_backend": "loky"}}
MONTH_WORKERS = MODE_SETTINGS[PROCESSING_MODE]["month_workers"]
PARALLEL_BACKEND = MODE_SETTINGS[PROCESSING_MODE]["parallel_backend"]
WORK_DIR = Path("/content/satpm_v6gl03_pm25_bolivia339" if IN_COLAB else "./satpm_v6gl03_pm25_bolivia339")
RAW_DIR = WORK_DIR / "raw_netcdf_cache"
TEMP_RASTER_DIR = WORK_DIR / "temporary_geotiffs"
CHECKPOINT_DIR = WORK_DIR / "checkpoints_yearly"
CHECKPOINT_DIR_YEARLY = CHECKPOINT_DIR
OUTPUT_DIR = WORK_DIR / "outputs"
FIGURE_DIR = WORK_DIR / "figures"
METADATA_DIR = WORK_DIR / "metadata"
MAP_DIR = WORK_DIR / "canonical_map"
for folder in [WORK_DIR, RAW_DIR, TEMP_RASTER_DIR, CHECKPOINT_DIR, OUTPUT_DIR, FIGURE_DIR, METADATA_DIR, MAP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
OUTPUT_STEM = f"SatPM_V6GL03_Monthly_PM25_Bolivia339_Municipality_Stats_{START_YEAR}_{END_YEAR}"
FINAL_PARQUET = OUTPUT_DIR / f"{OUTPUT_STEM}.parquet"
FINAL_CSV = OUTPUT_DIR / f"{OUTPUT_STEM}.csv"
FINAL_ZIP = OUTPUT_DIR / f"{OUTPUT_STEM}.zip"
ANNUAL_PARQUET = OUTPUT_DIR / f"{OUTPUT_STEM}_AnnualMeans.parquet"
RUN_METADATA_JSON = METADATA_DIR / f"{OUTPUT_STEM}_run_metadata.json"
ENABLE_HF_UPLOAD = True
OUTPUT_REPO_ID = "faviolc/gdo4Bolivia"
OUTPUT_REPO_TYPE = "dataset"
OUTPUT_HF_FOLDER = "pm25_monthly"
print("Bolivia 339 configuration loaded:", MAP_URL, OUTPUT_REPO_ID)

## Step 3. Hugging Face authentication

Configure `HF_Writing` in **Colab Secrets**. It must have write access to `faviolc/gdo4Bolivia`. Never embed the token in the notebook.

In [ ]:
def get_hf_token(secret_name=HF_TOKEN_SECRET_NAME):
    """Read a Hugging Face token from Colab Secrets or from environment variables."""
    token = None
    if IN_COLAB and userdata is not None:
        try:
            token = userdata.get(secret_name)
        except Exception:
            token = None
    if token is None:
        token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    return token

HF_TOKEN = get_hf_token()

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Hugging Face login completed.")
else:
    print("No Hugging Face token found. Public downloads may still work, but private files/uploads will fail.")

api = HfApi(token=HF_TOKEN)

## Step 4. Helper functions for inventory, filenames, and metadata

In [ ]:
def normalize_text(value):
    """Lowercase alphanumeric normalizer used for fuzzy column and variable matching."""
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())


def extract_year_month_from_text(text, start_year=START_YEAR, end_year=END_YEAR):
    """Extract a year and month from strings such as V6GL03.CNNPM25.SA.199901-199901.nc."""
    text = str(text)
    patterns = [
        r"(?<!\d)(19\d{2}|20\d{2})[\-_\. ]?(0[1-9]|1[0-2])(?!\d)",
        r"(?<!\d)(0[1-9]|1[0-2])[\-_\. ]?(19\d{2}|20\d{2})(?!\d)",
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            first, second = match.group(1), match.group(2)
            if len(first) == 4:
                year, month = int(first), int(second)
            else:
                month, year = int(first), int(second)
            if start_year <= year <= end_year:
                return year, month
    return None, None


def expected_pm25_hf_path(year, month, region_code=PM25_REGION_CODE):
    """Construct the expected Hugging Face path for one SatPM monthly regional NetCDF file."""
    return (
        f"Monthly/{region_code}/{year}/"
        f"V6GL03.CNNPM25.{region_code}.{year}{month:02d}-{year}{month:02d}.nc"
    )


def build_pm25_inventory_from_repo(repo_id=PM25_REPO_ID, repo_type=PM25_REPO_TYPE):
    """List NetCDF files in the PM2.5 Hugging Face repository and parse year/month metadata."""
    files = api.list_repo_files(repo_id=repo_id, repo_type=repo_type, token=HF_TOKEN)
    rows = []
    for path in files:
        if not path.startswith(PM25_BASE_FOLDER + "/"):
            continue
        if not path.lower().endswith(".nc"):
            continue
        year, month = extract_year_month_from_text(path)
        if year is None or month is None:
            continue
        filename = Path(path).name
        rows.append({
            "year": int(year),
            "month": int(month),
            "ym": f"{year}-{month:02d}",
            "date": pd.Timestamp(year=year, month=month, day=1),
            "filename": filename,
            "hf_path": path,
            "region_code": PM25_REGION_CODE,
        })
    inventory = pd.DataFrame(rows)
    if not inventory.empty:
        inventory = inventory.sort_values(["year", "month", "hf_path"]).reset_index(drop=True)
        inventory = inventory.drop_duplicates(subset=["year", "month"], keep="last").reset_index(drop=True)
    return inventory


def write_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
    print("Wrote:", path)

## Step 5. Inspect the PM₂.₅ Hugging Face inventory

This checks whether all expected South America monthly NetCDF files are visible in the repository.

In [ ]:
pm25_inventory = build_pm25_inventory_from_repo()

print(f"NetCDF files detected for {PM25_REGION_CODE}: {len(pm25_inventory):,}")
if not pm25_inventory.empty:
    print("Date range:", pm25_inventory["date"].min().date(), "to", pm25_inventory["date"].max().date())
    display(pm25_inventory.head(12))
    display(pm25_inventory.tail(12))
else:
    raise RuntimeError("No PM2.5 NetCDF files were detected. Check PM25_REPO_ID and PM25_BASE_FOLDER.")

expected_months = pd.MultiIndex.from_product(
    [range(START_YEAR, END_YEAR + 1), MONTHS],
    names=["year", "month"]
).to_frame(index=False)
coverage = expected_months.merge(
    pm25_inventory[["year", "month", "hf_path"]],
    on=["year", "month"],
    how="left"
)
missing_pm25 = coverage[coverage["hf_path"].isna()].copy()

print("Expected months:", len(expected_months))
print("Available months:", coverage["hf_path"].notna().sum())
print("Missing months:", len(missing_pm25))

if len(missing_pm25):
    display(missing_pm25)
else:
    print("Coverage is complete for the configured period.")

pm25_inventory_path = METADATA_DIR / f"pm25_inventory_{PM25_REGION_CODE}_{START_YEAR}_{END_YEAR}.csv"
pm25_inventory.to_csv(pm25_inventory_path, index=False)
print("Inventory saved to:", pm25_inventory_path)

## Step 6. Load and harmonize the **339 historical municipalities**

`asdf_id` links the historical GeoJSON to the accompanying lookup. `ID_ADMIN` is the zero-padded six-digit INE code; GADM and alternative IDs remain in every output.

In [ ]:
assert (START_YEAR, END_YEAR, EXPECTED_ADMIN_UNITS) == (1998, 2024, 339)
assert PM25_REPO_ID == "faviolc/monthlypm25"
assert OUTPUT_REPO_ID == "faviolc/gdo4Bolivia"
assert MAP_HF_PATH.endswith("bolivia339geoqueryOpt.geojson")
print("Canonical 339-unit Bolivia contract verified.")

In [ ]:
# Download historical GeoJSON + crosswalk from the same GitHub revision.
import requests, hashlib

def retrieve_checked(url, target, local_override=None):
    target = Path(target)
    if local_override:
        p = Path(local_override)
        if not p.is_file():
            raise FileNotFoundError(p)
        return p
    response = requests.get(url, timeout=90)
    response.raise_for_status()
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(response.content)
    print(target.name, "SHA256", hashlib.sha256(response.content).hexdigest())
    return target

map_path = retrieve_checked(MAP_URL, MAP_DIR / "bolivia339geoqueryOpt.geojson", LOCAL_ADMIN_FILE)
lookup_path = retrieve_checked(LOOKUP_URL, MAP_DIR / "municipalities339_gadm.csv", LOCAL_LOOKUP_FILE)
gdf_admin_raw = gpd.read_file(map_path)
lookup_raw = pd.read_csv(lookup_path, dtype={"ine_code": str, "GID_3": str})
print("Boundary rows:", len(gdf_admin_raw), "crosswalk rows:", len(lookup_raw))
print("GeoJSON columns:", list(gdf_admin_raw.columns))
display(gdf_admin_raw.head(3))

In [ ]:
def standardize_bolivia339(gdf_raw, crosswalk):
    if len(gdf_raw) != 339 or len(crosswalk) != 339:
        raise ValueError(f"Expected 339 boundary and lookup rows, got {len(gdf_raw)}, {len(crosswalk)}")
    if gdf_raw.crs is None:
        raise ValueError("Historical municipality GeoJSON has no CRS; do not assume one.")
    gdf = gdf_raw.to_crs("EPSG:4326").copy()
    required_shape = {"asdf_id", "geometry"}
    required_lookup = {"asdf_id", "ine_code", "GID_3", "mun_id", "name"}
    if not required_shape.issubset(gdf.columns) or not required_lookup.issubset(crosswalk.columns):
        raise ValueError(f"Missing columns in source; shape={list(gdf.columns)}, lookup={list(crosswalk.columns)}")
    gdf["asdf_id"] = pd.to_numeric(gdf["asdf_id"], errors="raise").astype("int64")
    xwalk = crosswalk.copy()
    xwalk["asdf_id"] = pd.to_numeric(xwalk["asdf_id"], errors="raise").astype("int64")
    if gdf["asdf_id"].duplicated().any() or xwalk["asdf_id"].duplicated().any():
        raise ValueError("Duplicate asdf_id in historical geography or lookup")
    if set(gdf["asdf_id"]) != set(xwalk["asdf_id"]):
        raise ValueError("339-map and lookup asdf_id sets do not match")
    # Validate one-to-one crosswalk rather than silently dropping polygons.
    out = gdf.merge(xwalk, on="asdf_id", how="left", validate="one_to_one", suffixes=("_shape", ""))
    out["ID_ADMIN"] = out["ine_code"].astype("string").str.zfill(6)
    out["ID_PROVINCE"] = out["ID_ADMIN"].str[:4]
    out["ID_DEPARTMENT"] = out["ID_ADMIN"].str[:2]
    out["municipality_name"] = out["name"].astype(str)
    out["gid_3"] = out["GID_3"].astype(str)
    out["mun_id"] = pd.to_numeric(out["mun_id"], errors="coerce").astype("Int64")
    if out["ID_ADMIN"].isna().any() or not out["ID_ADMIN"].str.fullmatch(r"\d{6}").all():
        raise ValueError("Invalid six-digit INE code")
    if out["ID_ADMIN"].nunique() != 339 or out["gid_3"].nunique() != 339:
        raise ValueError("Crosswalk is not one-to-one by INE and GADM")
    if out.geometry.isna().any() or out.geometry.is_empty.any() or not out.geometry.is_valid.all():
        raise ValueError("Missing/empty/invalid historical polygons; review source rather than silently fixing")
    out["area_km2"] = out.to_crs("EPSG:6933").geometry.area / 1e6
    out["map_repo_id"] = MAP_REPO_ID
    out["map_hf_path"] = MAP_HF_PATH
    out["map_revision"] = MAP_REVISION
    keep = ["ID_ADMIN", "ID_PROVINCE", "ID_DEPARTMENT", "municipality_name", "asdf_id", "mun_id", "gid_3", "area_km2", "map_repo_id", "map_hf_path", "map_revision", "geometry"]
    out = out[keep].sort_values("ID_ADMIN").reset_index(drop=True)
    return gpd.GeoDataFrame(out, geometry="geometry", crs="EPSG:4326")

gdf_admin = standardize_bolivia339(gdf_admin_raw, lookup_raw)
assert len(gdf_admin) == EXPECTED_ADMIN_UNITS
print("Bolivia historical municipalities:", len(gdf_admin))
display(gdf_admin.head())

In [ ]:
assert len(gdf_admin) == 339
assert gdf_admin["ID_ADMIN"].nunique() == 339
print("Historical municipality geography locked.")

## Step 7. Plot the historical **339 municipalities**

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
gdf_admin.boundary.plot(ax=ax, linewidth=0.2)
ax.set_title("Bolivia 339 historical municipalities used for PM2.5 zonal statistics")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

## Step 8. NetCDF inspection and conversion helpers

These functions defensively identify the PM₂.₅ variable, latitude, longitude, CRS, and raster orientation. Each monthly NetCDF is converted to a temporary GeoTIFF before zonal statistics are computed.

In [ ]:
def guess_pm25_variable(data_vars):
    """
    Identify the PM2.5 variable conservatively.

    Production must fail rather than silently selecting an unrelated numeric
    variable from the NetCDF file.
    """
    accepted_exact = {
        "PM25", "pm25", "PM2_5", "pm2_5", "PM2.5",
        "CNNPM25", "cnnpm25",
    }

    exact_matches = [name for name in data_vars if name in accepted_exact]
    semantic_matches = [
        name for name in data_vars
        if normalize_text(name) in {"pm25", "pm2p5", "cnnpm25"}
    ]

    matches = list(dict.fromkeys(exact_matches + semantic_matches))

    if len(matches) != 1:
        raise ValueError(
            "Expected exactly one recognized PM2.5 variable. "
            f"Recognized={matches}; available={list(data_vars)}"
        )
    return matches[0]


def guess_lat_lon_names(ds, da):
    """Guess latitude and longitude coordinate names from dataset and data array metadata."""
    names = list(dict.fromkeys(list(da.dims) + list(da.coords) + list(ds.coords) + list(ds.dims)))
    lat_candidates = []
    lon_candidates = []
    for name in names:
        norm = normalize_text(name)
        if norm in ["lat", "latitude", "y"] or "latitude" in norm:
            lat_candidates.append(name)
        if norm in ["lon", "longitude", "x"] or "longitude" in norm:
            lon_candidates.append(name)
    lat_name = lat_candidates[0] if lat_candidates else None
    lon_name = lon_candidates[0] if lon_candidates else None
    return lat_name, lon_name


def open_dataset_safely(nc_path):
    """Open a NetCDF file with several engines, returning the dataset and engine name."""
    engines = ["h5netcdf", "netcdf4", None]
    last_error = None
    for engine in engines:
        try:
            kwargs = {"mask_and_scale": True, "decode_times": False}
            if engine is not None:
                kwargs["engine"] = engine
            ds = xr.open_dataset(nc_path, **kwargs)
            return ds, (engine or "default")
        except Exception as error:
            last_error = error
    raise RuntimeError(f"Could not open NetCDF file {nc_path}: {last_error}")


def describe_netcdf(nc_path):
    """Return lightweight metadata for one NetCDF file without forcing all values into memory."""
    ds, engine = open_dataset_safely(nc_path)
    try:
        data_vars = list(ds.data_vars)
        pm25_variable = guess_pm25_variable(data_vars)
        da = ds[pm25_variable]

        # Inspect source units instead of merely assigning them downstream.
        source_units = str(da.attrs.get("units", "")).strip()
        if source_units:
            normalized_units = normalize_text(source_units)
            accepted_unit_tokens = {
                "ugm3", "microgramperm3", "microgramspercubicmeter",
                "microgrampercubicmeter", "gm3"
            }
            if normalized_units not in accepted_unit_tokens:
                print(
                    "Warning: unexpected PM2.5 source units:",
                    source_units,
                    "— inspect the smoke-test metadata before production."
                )
        else:
            print("Warning: PM2.5 NetCDF variable has no units attribute.")
        lat_name, lon_name = guess_lat_lon_names(ds, da)
        attrs = {k: str(v) for k, v in da.attrs.items()}
        summary = {
            "path": str(nc_path),
            "engine": engine,
            "dims": {str(k): int(v) for k, v in ds.sizes.items()},
            "data_vars": data_vars,
            "pm25_variable_guess": pm25_variable,
            "lat_name_guess": lat_name,
            "lon_name_guess": lon_name,
            "pm25_attrs": attrs,
        }
        return summary
    finally:
        ds.close()


def prepare_pm25_dataarray(nc_path, pm25_variable=None):
    """
    Read a monthly PM2.5 NetCDF as a 2D DataArray ready for GeoTIFF export.

    The returned array is loaded into memory and detached from the xarray Dataset,
    so the underlying NetCDF file can be closed safely.
    """
    ds, engine = open_dataset_safely(nc_path)
    try:
        data_vars = list(ds.data_vars)
        if pm25_variable is None:
            pm25_variable = guess_pm25_variable(data_vars)
        if pm25_variable not in ds.data_vars:
            raise ValueError(f"Variable {pm25_variable!r} not found. Available variables: {data_vars}")

        da = ds[pm25_variable]
        lat_name, lon_name = guess_lat_lon_names(ds, da)
        if lat_name is None or lon_name is None:
            raise ValueError(
                f"Could not identify latitude/longitude names. dims={list(da.dims)}, coords={list(da.coords)}"
            )

        # Select the first value of any non-spatial dimension. Monthly files should
        # usually have no time dimension or a singleton time dimension.
        spatial_names = {lat_name, lon_name}
        for dim in list(da.dims):
            if dim not in spatial_names:
                da = da.isel({dim: 0}, drop=True)

        # If latitude/longitude are coordinates with different dimension names,
        # swap dimensions so rioxarray can understand the spatial axes.
        if lat_name not in da.dims and lat_name in da.coords and len(da[lat_name].dims) == 1:
            da = da.swap_dims({da[lat_name].dims[0]: lat_name})
        if lon_name not in da.dims and lon_name in da.coords and len(da[lon_name].dims) == 1:
            da = da.swap_dims({da[lon_name].dims[0]: lon_name})

        da = da.squeeze(drop=True)
        if len(da.dims) != 2:
            raise ValueError(f"Expected a 2D PM2.5 array after squeezing, got dims={da.dims}")

        da = da.astype("float32")

        # Convert longitudes from 0..360 to -180..180 if needed.
        lon_values = da[lon_name].values
        if np.nanmax(lon_values) > 180:
            new_lon = ((lon_values + 180) % 360) - 180
            da = da.assign_coords({lon_name: new_lon}).sortby(lon_name)

        # Force north-up orientation. Many NetCDF grids store latitude ascending.
        lat_values = da[lat_name].values
        if len(lat_values) > 1 and lat_values[0] < lat_values[-1]:
            da = da.sortby(lat_name, ascending=False)

        da = da.where(np.isfinite(da))
        if PM25_NEGATIVE_TO_NODATA:
            da = da.where(da >= 0)

        da = da.rio.set_spatial_dims(x_dim=lon_name, y_dim=lat_name, inplace=False)
        da = da.rio.write_crs("EPSG:4326", inplace=False)
        da = da.rio.write_nodata(RASTER_NODATA, inplace=False)
        da = da.load()

        metadata = {
            "engine": engine,
            "pm25_variable": pm25_variable,
            "lat_name": lat_name,
            "lon_name": lon_name,
            "shape": tuple(int(x) for x in da.shape),
            "min": float(np.nanmin(da.values)) if np.isfinite(da.values).any() else None,
            "max": float(np.nanmax(da.values)) if np.isfinite(da.values).any() else None,
            "mean": float(np.nanmean(da.values)) if np.isfinite(da.values).any() else None,
            "negative_to_nodata": PM25_NEGATIVE_TO_NODATA,
        }
        return da, metadata
    finally:
        ds.close()


def write_pm25_geotiff_from_netcdf(nc_path, tif_path, pm25_variable=None):
    """Convert one monthly NetCDF to a temporary EPSG:4326 GeoTIFF."""
    tif_path = Path(tif_path)
    tif_path.parent.mkdir(parents=True, exist_ok=True)
    da, metadata = prepare_pm25_dataarray(nc_path, pm25_variable=pm25_variable)
    da_to_write = da.fillna(RASTER_NODATA).rio.write_nodata(RASTER_NODATA, inplace=False)
    da_to_write.rio.to_raster(
        tif_path,
        dtype="float32",
        compress="DEFLATE",
        predictor=3,
        tiled=True,
    )
    metadata["temporary_geotiff"] = str(tif_path)
    return tif_path, metadata


def count_zero(masked_array):
    arr = np.ma.filled(masked_array, np.nan)
    return int(np.nansum(arr == 0))


def count_positive(masked_array):
    arr = np.ma.filled(masked_array, np.nan)
    return int(np.nansum(arr > 0))

## Step 9. Smoke test: download and inspect one monthly NetCDF

The smoke test should be run before any production loop. It verifies the file path, NetCDF engine, PM₂.₅ variable name, coordinate names, and array dimensions.

In [ ]:
SMOKE_TEST_YEAR = 1999
SMOKE_TEST_MONTH = 1

smoke_row = pm25_inventory.query("year == @SMOKE_TEST_YEAR and month == @SMOKE_TEST_MONTH")
if smoke_row.empty:
    raise ValueError(f"Smoke-test file not found for {SMOKE_TEST_YEAR}-{SMOKE_TEST_MONTH:02d}.")
smoke_row = smoke_row.iloc[0]

smoke_nc_path = hf_hub_download(
    repo_id=PM25_REPO_ID,
    repo_type=PM25_REPO_TYPE,
    filename=smoke_row["hf_path"],
    token=HF_TOKEN,
    local_dir=RAW_DIR,
    local_dir_use_symlinks=False,
)
smoke_nc_path = Path(smoke_nc_path)
print("Downloaded smoke-test NetCDF:", smoke_nc_path)
print("File size MB:", smoke_nc_path.stat().st_size / (1024 ** 2))

smoke_nc_summary = describe_netcdf(smoke_nc_path)
print(json.dumps(smoke_nc_summary, indent=2, default=str))

## Step 10. Smoke test: convert one NetCDF to GeoTIFF and inspect bounds

In [ ]:
# ------------------------------------------------------------------------------
# Robust NetCDF PM2.5 -> GeoTIFF writer
# ------------------------------------------------------------------------------

import json
import numpy as np
import xarray as xr
import rasterio
from rasterio.transform import from_origin
from pathlib import Path


def find_pm25_variable(ds):
    """Find the PM2.5 variable inside the NetCDF file."""

    preferred_names = [
        "PM25",
        "pm25",
        "PM2_5",
        "PM2.5",
        "CNNPM25",
        "cnnpm25",
    ]

    for name in preferred_names:
        if name in ds.data_vars:
            return name

    # Fallback: choose the first numeric data variable with at least 2 dimensions.
    for name, da in ds.data_vars.items():
        if np.issubdtype(da.dtype, np.number) and len(da.dims) >= 2:
            return name

    raise ValueError(
        "Could not identify a PM2.5 variable. "
        f"Available data variables: {list(ds.data_vars)}"
    )


def find_lat_lon_dims(da, ds):
    """Find latitude and longitude dimension names."""

    lat_candidates = ["lat", "latitude", "Latitude", "LAT", "y", "Y"]
    lon_candidates = ["lon", "longitude", "Longitude", "LON", "x", "X"]

    lat_dim = None
    lon_dim = None

    # First search directly in data-array dimensions.
    for dim in da.dims:
        if dim in lat_candidates:
            lat_dim = dim
        if dim in lon_candidates:
            lon_dim = dim

    # If still missing, inspect coordinates.
    if lat_dim is None:
        for coord in ds.coords:
            if coord in lat_candidates and len(ds[coord].dims) == 1:
                candidate_dim = ds[coord].dims[0]
                if candidate_dim in da.dims:
                    lat_dim = candidate_dim
                    break

    if lon_dim is None:
        for coord in ds.coords:
            if coord in lon_candidates and len(ds[coord].dims) == 1:
                candidate_dim = ds[coord].dims[0]
                if candidate_dim in da.dims:
                    lon_dim = candidate_dim
                    break

    if lat_dim is None or lon_dim is None:
        raise ValueError(
            "Could not identify latitude/longitude dimensions.\n"
            f"Data variable dims: {da.dims}\n"
            f"Dataset coords: {list(ds.coords)}\n"
            f"Dataset dims: {dict(ds.sizes)}"
        )

    return lat_dim, lon_dim


def get_coord_for_dim(ds, dim_name, coord_candidates):
    """Return the coordinate array associated with a dimension."""

    # Best case: dimension itself is also a coordinate.
    if dim_name in ds.coords:
        return ds[dim_name].values

    # Otherwise search common coordinate names.
    for coord in coord_candidates:
        if coord in ds.coords and ds[coord].dims == (dim_name,):
            return ds[coord].values

    raise ValueError(
        f"Could not find coordinate values for dimension '{dim_name}'. "
        f"Available coords: {list(ds.coords)}"
    )


def write_pm25_geotiff_from_netcdf(netcdf_path, output_tif_path, variable_name=None):
    """
    Convert one monthly PM2.5 NetCDF file to a temporary GeoTIFF.

    This version avoids rioxarray spatial-dimension errors by:
    - detecting the PM2.5 variable,
    - detecting lat/lon dimensions,
    - sorting longitude ascending and latitude descending,
    - writing the GeoTIFF manually with rasterio.
    """

    netcdf_path = Path(netcdf_path)
    output_tif_path = Path(output_tif_path)
    output_tif_path.parent.mkdir(parents=True, exist_ok=True)

    ds = xr.open_dataset(netcdf_path)

    try:
        if variable_name is None:
            variable_name = find_pm25_variable(ds)

        da = ds[variable_name]

        # Remove singleton non-spatial dimensions, e.g. time=1.
        da = da.squeeze(drop=True)

        lat_dim, lon_dim = find_lat_lon_dims(da, ds)

        lat_values = get_coord_for_dim(
            ds,
            lat_dim,
            ["lat", "latitude", "Latitude", "LAT", "y", "Y"],
        )

        lon_values = get_coord_for_dim(
            ds,
            lon_dim,
            ["lon", "longitude", "Longitude", "LON", "x", "X"],
        )

        lat_values = np.asarray(lat_values).astype("float64")
        lon_values = np.asarray(lon_values).astype("float64")

        # Keep only the two spatial dimensions in y/x order.
        da = da.transpose(lat_dim, lon_dim)

        arr = da.values.astype("float32")

        # Sort longitude ascending: west -> east.
        lon_order = np.argsort(lon_values)
        lon_values = lon_values[lon_order]
        arr = arr[:, lon_order]

        # Sort latitude descending: north -> south.
        lat_order = np.argsort(lat_values)[::-1]
        lat_values = lat_values[lat_order]
        arr = arr[lat_order, :]

        # Convert fill values and invalid values to nodata.
        nodata_value = -9999.0

        fill_candidates = []
        for key in ["_FillValue", "missing_value"]:
            if key in da.attrs:
                fill_candidates.append(da.attrs[key])
            if key in da.encoding:
                fill_candidates.append(da.encoding[key])

        for fill in fill_candidates:
            try:
                arr[arr == float(fill)] = np.nan
            except Exception:
                pass

        arr[~np.isfinite(arr)] = np.nan

        # PM2.5 should not be negative. Keep this conservative.
        arr[arr < 0] = np.nan

        valid_mask = np.isfinite(arr)
        arr_to_write = arr.copy()
        arr_to_write[~valid_mask] = nodata_value

        # Estimate resolution from coordinate spacing.
        lon_res = float(np.median(np.abs(np.diff(lon_values))))
        lat_res = float(np.median(np.abs(np.diff(lat_values))))

        west = float(lon_values.min() - lon_res / 2)
        north = float(lat_values.max() + lat_res / 2)

        transform = from_origin(west, north, lon_res, lat_res)

        height, width = arr_to_write.shape

        metadata = {
            "netcdf_path": str(netcdf_path),
            "output_tif_path": str(output_tif_path),
            "variable_name": variable_name,
            "original_dims_after_squeeze": tuple(da.dims),
            "lat_dim": lat_dim,
            "lon_dim": lon_dim,
            "height": height,
            "width": width,
            "lon_min": float(lon_values.min()),
            "lon_max": float(lon_values.max()),
            "lat_min": float(lat_values.min()),
            "lat_max": float(lat_values.max()),
            "lon_res": lon_res,
            "lat_res": lat_res,
            "nodata": nodata_value,
            "valid_pixels": int(valid_mask.sum()),
            "nan_pixels": int((~valid_mask).sum()),
            "pm25_min": float(np.nanmin(arr)) if valid_mask.any() else None,
            "pm25_max": float(np.nanmax(arr)) if valid_mask.any() else None,
            "pm25_mean": float(np.nanmean(arr)) if valid_mask.any() else None,
        }

        with rasterio.open(
            output_tif_path,
            "w",
            driver="GTiff",
            height=height,
            width=width,
            count=1,
            dtype="float32",
            crs="EPSG:4326",
            transform=transform,
            nodata=nodata_value,
            compress="deflate",
        ) as dst:
            dst.write(arr_to_write, 1)

        return output_tif_path, metadata

    finally:
        ds.close()

In [ ]:
smoke_tif_path = TEMP_RASTER_DIR / f"smoke_pm25_{SMOKE_TEST_YEAR}{SMOKE_TEST_MONTH:02d}.tif"
smoke_tif_path, smoke_raster_metadata = write_pm25_geotiff_from_netcdf(smoke_nc_path, smoke_tif_path)

print(json.dumps(smoke_raster_metadata, indent=2, default=str))

with rasterio.open(smoke_tif_path) as src:
    print("Raster CRS:", src.crs)
    print("Raster bounds:", src.bounds)
    print("Raster shape:", src.height, src.width)
    print("Raster nodata:", src.nodata)
    raster_bounds = box(*src.bounds)

admin_bounds = box(*gdf_admin.total_bounds)
print("Raster intersects Bolivia municipality bounds:", raster_bounds.intersects(admin_bounds))

fig, ax = plt.subplots(figsize=(8, 8))
with rasterio.open(smoke_tif_path) as src:
    arr = src.read(1).astype("float32")
    arr[arr == src.nodata] = np.nan
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
    im = ax.imshow(arr, extent=extent, origin="upper")
    gdf_admin.boundary.plot(ax=ax, linewidth=0.15)
    plt.colorbar(im, ax=ax, shrink=0.7, label="PM2.5")
ax.set_title(f"Smoke-test PM2.5 raster and Bolivia municipalities: {SMOKE_TEST_YEAR}-{SMOKE_TEST_MONTH:02d}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------------------
# Final zonal-statistics implementation:
# percentiles + IQR + fallback + sum/count/mean audit
# ------------------------------------------------------------------------------

def _valid_values(masked_array):
    values = np.ma.compressed(masked_array)
    values = np.asarray(values, dtype="float64")
    return values[np.isfinite(values)]


def percentile_stat(q):
    def _stat(masked_array):
        values = _valid_values(masked_array)
        return float(np.percentile(values, q)) if values.size else np.nan
    _stat.__name__ = f"p{q}"
    return _stat


def iqr_stat(masked_array):
    values = _valid_values(masked_array)
    if not values.size:
        return np.nan
    return float(np.percentile(values, 75) - np.percentile(values, 25))


EXTRA_STATS = {
    "p10": percentile_stat(10),
    "p25": percentile_stat(25),
    "p75": percentile_stat(75),
    "p90": percentile_stat(90),
    "iqr": iqr_stat,
    "count_zero": count_zero,
    "count_positive": count_positive,
}


def _raw_zonal_stats(tif_path, gdf, all_touched):
    zs = zonal_stats(
        vectors=gdf.geometry,
        raster=str(tif_path),
        stats=BASE_STATS,
        add_stats=EXTRA_STATS,
        nodata=RASTER_NODATA,
        all_touched=all_touched,
        geojson_out=False,
    )
    stats_df = pd.DataFrame(zs)
    return stats_df.rename(columns={col: f"pm25_{col}" for col in stats_df.columns})


def run_zonal_stats_for_geotiff(tif_path, gdf, all_touched=ALL_TOUCHED):
    """
    Calculate municipality PM2.5 statistics.

    Main rule:
    - all_touched=False (pixel-center rule).

    Fallback:
    - municipalities with no valid centered pixel are recalculated with
      all_touched=True.

    Audit:
    - pm25_sum / pm25_count must reproduce pm25_mean within floating-point
      tolerance.
    """

    stats_df = _raw_zonal_stats(tif_path, gdf, all_touched=all_touched)

    stats_df["pm25_used_all_touched_fallback"] = False
    stats_df["pm25_count_strict"] = stats_df["pm25_count"]
    stats_df["pm25_mean_strict"] = stats_df["pm25_mean"]

    needs_fallback = (
        stats_df["pm25_mean"].isna()
        | stats_df["pm25_count"].isna()
        | (stats_df["pm25_count"] == 0)
    )

    if needs_fallback.any():
        n_fallback = int(needs_fallback.sum())
        print(f"Fallback all_touched=True needed for {n_fallback:,} municipality(s).")

        fallback_gdf = gdf.loc[needs_fallback].copy()
        fallback_df = _raw_zonal_stats(
            tif_path,
            fallback_gdf,
            all_touched=True,
        )
        fallback_index = stats_df.index[needs_fallback]

        stats_df.loc[fallback_index, "pm25_count_fallback"] = fallback_df[
            "pm25_count"
        ].to_numpy()
        stats_df.loc[fallback_index, "pm25_mean_fallback"] = fallback_df[
            "pm25_mean"
        ].to_numpy()

        for col in fallback_df.columns:
            stats_df.loc[fallback_index, col] = fallback_df[col].to_numpy()

        stats_df.loc[
            fallback_index, "pm25_used_all_touched_fallback"
        ] = True

    # Compute the mean explicitly from the sum and the number of valid raster cells.
    valid_count = stats_df["pm25_count"].notna() & (stats_df["pm25_count"] > 0)
    stats_df["pm25_valid_cell_count"] = stats_df["pm25_count"]
    stats_df["pm25_mean_valid_cells"] = np.nan
    stats_df.loc[valid_count, "pm25_mean_valid_cells"] = (
        stats_df.loc[valid_count, "pm25_sum"]
        / stats_df.loc[valid_count, "pm25_count"]
    )
    # `pm25_count` returned by rasterstats is the number of non-nodata
    # cells used in the zonal calculation. Therefore this is the requested
    # average over valid cells:
    #
    #     pm25_sum / pm25_valid_cell_count
    #
    # Keep the library-produced `pm25_mean` as a reference, but do not stop
    # production or perform a redundant equality assertion.


    return stats_df


def add_month_metadata(df, row, raster_metadata=None):
    """Attach time, source, raster, and canonical-map metadata."""

    out = df.copy()
    out.insert(0, "year", int(row["year"]))
    out.insert(1, "month", int(row["month"]))
    out.insert(
        2,
        "date",
        pd.Timestamp(year=int(row["year"]), month=int(row["month"]), day=1),
    )
    out.insert(3, "ym", f"{int(row['year'])}-{int(row['month']):02d}")

    out["source_repo"] = PM25_REPO_ID
    out["source_hf_path"] = row["hf_path"]
    out["product_name"] = PRODUCT_NAME
    out["pm25_units"] = PM25_UNITS
    out["all_touched_default"] = ALL_TOUCHED
    out["map_repo_id"] = MAP_REPO_ID
    out["map_hf_path"] = MAP_HF_PATH
    out["map_revision"] = MAP_REVISION

    if raster_metadata is not None:
        out["raster_pm25_variable"] = raster_metadata.get("variable_name")
        out["raster_lat_name"] = raster_metadata.get("lat_dim")
        out["raster_lon_name"] = raster_metadata.get("lon_dim")
        out["raster_engine"] = "xarray_to_rasterio_manual"
        out["raster_shape"] = (
            f"{raster_metadata.get('height')}x{raster_metadata.get('width')}"
        )
        out["raster_min"] = raster_metadata.get("pm25_min")
        out["raster_max"] = raster_metadata.get("pm25_max")
        out["raster_mean"] = raster_metadata.get("pm25_mean")

    return out


def validate_year_panel(year_df, year):
    """Validate one yearly checkpoint including distribution and fallback QA."""

    expected_rows = len(gdf_admin) * 12
    out = {
        "year": int(year),
        "rows": int(len(year_df)),
        "expected_rows": int(expected_rows),
        "unique_municipalities": int(year_df["ID_ADMIN"].nunique()),
        "unique_months": int(year_df["month"].nunique()),
        "duplicate_municipality_month_rows": int(
            year_df.duplicated(["ID_ADMIN", "year", "month"]).sum()
        ),
        "missing_pm25_mean": int(year_df["pm25_mean"].isna().sum()),
        "zero_pm25_count": int((year_df["pm25_count"] == 0).sum()),
        "fallback_rows": int(
            year_df["pm25_used_all_touched_fallback"].sum()
        ),
    }

    counts_by_month = (
        year_df.groupby("month")["ID_ADMIN"].nunique().to_dict()
    )
    out["municipalities_by_month"] = {
        int(k): int(v) for k, v in counts_by_month.items()
    }

    out["is_clean"] = (
        out["rows"] == out["expected_rows"]
        and out["unique_municipalities"] == EXPECTED_ADMIN_UNITS
        and out["unique_months"] == 12
        and out["duplicate_municipality_month_rows"] == 0
        and out["missing_pm25_mean"] == 0
        and out["zero_pm25_count"] == 0
    )
    return out

print("Final PM2.5 zonal-statistics implementation loaded.")


## Step 11. Smoke test: 339-municipality monthly PM₂.₅ statistics

Missing raster coverage is reported, not silently filled. Review any unresolved municipalities.

In [ ]:
# Step 11 smoke test using the corrected zonal-statistics implementation.

def admin_attributes_without_geometry(gdf):
    cols = [c for c in gdf.columns if c != "geometry"]
    return gdf[cols].reset_index(drop=True).copy()


smoke_stats = run_zonal_stats_for_geotiff(
    smoke_tif_path,
    gdf_admin,
    all_touched=ALL_TOUCHED,
)

smoke_test = pd.concat(
    [
        admin_attributes_without_geometry(gdf_admin),
        smoke_stats.reset_index(drop=True),
    ],
    axis=1,
)

smoke_test = add_month_metadata(
    smoke_test,
    smoke_row,
    smoke_raster_metadata,
)

required_corrected_columns = {
    "pm25_valid_cell_count",
    "pm25_mean_valid_cells",
    "pm25_used_all_touched_fallback",
}
missing_corrected_columns = required_corrected_columns - set(smoke_test.columns)
if missing_corrected_columns:
    raise RuntimeError(
        "The corrected zonal-statistics implementation was not used. "
        f"Missing columns: {sorted(missing_corrected_columns)}"
    )

unresolved_mask = (
    smoke_test["pm25_mean_valid_cells"].isna()
    | smoke_test["pm25_valid_cell_count"].fillna(0).eq(0)
)

print("Smoke-test rows:", len(smoke_test))
print("Expected rows:", len(gdf_admin))
print("Missing valid-cell PM2.5 means:", int(unresolved_mask.sum()))
print(
    "Fallback rows:",
    int(smoke_test["pm25_used_all_touched_fallback"].fillna(False).sum()),
)

display(smoke_test.head())

display(
    smoke_test[
        [
            "pm25_mean_valid_cells",
            "pm25_median",
            "pm25_min",
            "pm25_max",
            "pm25_std",
            "pm25_valid_cell_count",
        ]
    ].describe()
)

if unresolved_mask.any():
    unresolved_smoke = smoke_test.loc[
        unresolved_mask,
        [
            "ID_ADMIN",
            "ID_DEPARTMENT",
            "ID_PROVINCE",
            "department_name",
            "province_name",
            "municipality_name",
            "area_km2",
            "pm25_valid_cell_count",
            "pm25_sum",
            "pm25_mean_valid_cells",
            "pm25_used_all_touched_fallback",
        ],
    ]
    print("Municipalities still unresolved after fallback:")
    display(unresolved_smoke)
else:
    print("All 1,793 municipalities have valid PM2.5 values after fallback.")


## Step 12. Core monthly and yearly processing functions

In [ ]:
def process_one_month(row_dict, admin_gdf=None, cleanup_temporary=True):
    """Download, convert, and summarize one monthly NetCDF file."""
    if admin_gdf is None:
        admin_gdf = gdf_admin

    year = int(row_dict["year"])
    month = int(row_dict["month"])
    ym = f"{year}{month:02d}"
    month_temp_dir = TEMP_RASTER_DIR / ym
    month_temp_dir.mkdir(parents=True, exist_ok=True)
    tif_path = month_temp_dir / f"pm25_{ym}.tif"

    try:
        nc_path = hf_hub_download(
            repo_id=PM25_REPO_ID,
            repo_type=PM25_REPO_TYPE,
            filename=row_dict["hf_path"],
            token=HF_TOKEN,
            local_dir=RAW_DIR,
            local_dir_use_symlinks=False,
        )
        nc_path = Path(nc_path)

        tif_path, raster_metadata = write_pm25_geotiff_from_netcdf(nc_path, tif_path)
        stats_df = run_zonal_stats_for_geotiff(tif_path, admin_gdf)
        month_panel = pd.concat([admin_attributes_without_geometry(admin_gdf), stats_df], axis=1)
        month_panel = add_month_metadata(month_panel, row_dict, raster_metadata)

        # Stable column order: time, ID, names, stats, source metadata.
        front_cols = ["year", "month", "date", "ym", "ID_ADMIN"]
        name_cols = [c for c in ["department_name", "province_name", "municipality_name"] if c in month_panel.columns]
        stat_cols = [c for c in month_panel.columns if c.startswith("pm25_")]
        source_cols = [c for c in month_panel.columns if c not in front_cols + name_cols + stat_cols + ["area_deg2"]]
        ordered_cols = front_cols + name_cols + ["area_deg2"] + stat_cols + source_cols
        ordered_cols = [c for c in ordered_cols if c in month_panel.columns]
        month_panel = month_panel[ordered_cols].copy()

        return month_panel

    except Exception as error:
        print(f"Failed month {year}-{month:02d}: {error}")
        traceback.print_exc()
        raise

    finally:
        if cleanup_temporary:
            shutil.rmtree(month_temp_dir, ignore_errors=True)


def validate_year_panel(year_df, year):
    """Validate one yearly checkpoint."""
    expected_rows = len(gdf_admin) * 12
    out = {
        "year": int(year),
        "rows": int(len(year_df)),
        "expected_rows": int(expected_rows),
        "unique_municipalities": int(year_df["ID_ADMIN"].nunique()),
        "unique_months": int(year_df["month"].nunique()),
        "duplicate_municipality_month_rows": int(year_df.duplicated(["ID_ADMIN", "year", "month"]).sum()),
        "missing_pm25_mean": int(year_df["pm25_mean"].isna().sum()) if "pm25_mean" in year_df.columns else None,
    }
    counts_by_month = year_df.groupby("month")["ID_ADMIN"].nunique().to_dict()
    out["municipalities_by_month"] = {int(k): int(v) for k, v in counts_by_month.items()}
    return out


def process_year(year, overwrite=False, month_workers=MONTH_WORKERS):
    """Process all 12 months for one year and save a yearly checkpoint."""
    year = int(year)
    checkpoint_path = CHECKPOINT_DIR / f"SatPM_V6GL03_PM25_Bolivia_Municipality_Stats_{year}.parquet"
    validation_path = CHECKPOINT_DIR / f"SatPM_V6GL03_PM25_Bolivia_Municipality_Stats_{year}_validation.json"

    if checkpoint_path.exists() and not overwrite:
        print(f"Checkpoint already exists for {year}. Reading: {checkpoint_path}")
        return pd.read_parquet(checkpoint_path)

    year_inventory = pm25_inventory.query("year == @year").sort_values("month").copy()
    if len(year_inventory) != 12:
        raise ValueError(f"Expected 12 monthly files for {year}, found {len(year_inventory)}.")

    row_dicts = year_inventory.to_dict("records")

    if month_workers == 1:
        monthly_panels = [process_one_month(row) for row in tqdm(row_dicts, desc=f"Processing {year}")]
    else:
        monthly_panels = Parallel(
            n_jobs=month_workers,
            backend=PARALLEL_BACKEND,
            verbose=10,
        )(
            delayed(process_one_month)(row) for row in row_dicts
        )

    year_df = pd.concat(monthly_panels, ignore_index=True)
    year_df = year_df.sort_values(["year", "month", "ID_ADMIN"]).reset_index(drop=True)

    year_df.to_parquet(checkpoint_path, index=False)
    validation = validate_year_panel(year_df, year)
    write_json(validation, validation_path)

    print(f"Saved yearly checkpoint: {checkpoint_path}")
    print(json.dumps(validation, indent=2, default=str))
    return year_df

## Step 13. One-year qualification gate

The smoke test runs first. Processing the entire 1998–2024 period is controlled by a separate explicit toggle.

In [ ]:
if "smoke_test" not in globals():
    raise RuntimeError("Smoke test must run first.")
assert len(smoke_test) == 339
unresolved_count = int(smoke_test["pm25_mean_valid_cells"].isna().sum())
print("Unresolved municipality raster means:", unresolved_count)
if unresolved_count:
    print("Investigate raster coverage. Missing values must remain NaN; do not zero-fill.")

In [ ]:
RUN_ONE_YEAR_TEST = True
TEST_YEAR = 1999
OVERWRITE_TEST_YEAR = False
if RUN_ONE_YEAR_TEST:
    test_year_df = process_year(TEST_YEAR, overwrite=OVERWRITE_TEST_YEAR, month_workers=MONTH_WORKERS)
    print("Rows:", len(test_year_df), "Expected:", 339 * 12)
    assert len(test_year_df) == 339 * 12
    assert not test_year_df.duplicated(["ID_ADMIN", "year", "month"]).any()
    display(test_year_df.head())
else:
    print("One-year qualification disabled.")

In [ ]:
# Optional diagnostics on the one-year checkpoint.
if RUN_ONE_YEAR_TEST:
    print("Missing means:", int(test_year_df["pm25_mean_valid_cells"].isna().sum()))
    display(test_year_df.groupby("month")["ID_ADMIN"].nunique())

In [ ]:
# This original Peru debugging cell is intentionally retired for Bolivia.
print("Use the one-year diagnostic cell above; no forced recomputation.")

## Step 14. Production loop for 1998–2024

This section processes years sequentially and months within each year in parallel. It saves one checkpoint per year, so the run can be resumed safely.

In [ ]:
# ------------------------------------------------------------------------------
# Full production loop with yearly Parquet export, Hugging Face upload,
# and temporary GeoTIFF cleanup
# ------------------------------------------------------------------------------

from pathlib import Path
import os
import json
import pandas as pd
from tqdm.auto import tqdm
from huggingface_hub import HfApi

# ------------------------------------------------------------------------------
# Production switches
# ------------------------------------------------------------------------------

RUN_PRODUCTION = True

FULL_START_YEAR = 1998
FULL_END_YEAR = 2024

YEARS_TO_PROCESS = list(range(FULL_START_YEAR, FULL_END_YEAR + 1))

OVERWRITE_YEARLY_CHECKPOINTS = False
MONTH_WORKERS = 2

UPLOAD_YEARLY_TO_HF = True
DELETE_TEMP_GEOTIFFS_AFTER_UPLOAD = True

# ------------------------------------------------------------------------------
# Correct Hugging Face output destination
# ------------------------------------------------------------------------------

OUTPUT_HF_REPO_ID = "faviolc/gdo4Bolivia"
OUTPUT_HF_REPO_TYPE = "dataset"
OUTPUT_HF_BASE_FOLDER = "pm25_monthly"

# ------------------------------------------------------------------------------
# Safe path fallbacks
# ------------------------------------------------------------------------------

BASE_DIR = Path(
    globals().get(
        "BASE_DIR",
        "/content/satpm_v6gl03_pm25_bolivia339",
    )
)

CHECKPOINT_DIR_YEARLY = Path(
    globals().get(
        "CHECKPOINT_DIR_YEARLY",
        BASE_DIR / "checkpoints_yearly",
    )
)

METADATA_DIR = Path(
    globals().get(
        "METADATA_DIR",
        BASE_DIR / "metadata",
    )
)

TEMP_RASTER_DIR = Path(
    globals().get(
        "TEMP_RASTER_DIR",
        BASE_DIR / "temporary_geotiffs",
    )
)

CHECKPOINT_DIR_YEARLY.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)
TEMP_RASTER_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# Hugging Face token
# ------------------------------------------------------------------------------

HF_UPLOAD_TOKEN = None

for token_name in ["HF_TOKEN", "VECTOR_HF_TOKEN"]:
    if token_name in globals() and globals()[token_name] is not None:
        HF_UPLOAD_TOKEN = globals()[token_name]
        break

if HF_UPLOAD_TOKEN is None:
    try:
        from google.colab import userdata

        HF_UPLOAD_TOKEN = userdata.get("HF_Writing")
    except Exception:
        pass

if HF_UPLOAD_TOKEN is None:
    HF_UPLOAD_TOKEN = os.environ.get("HF_TOKEN")

if HF_UPLOAD_TOKEN is None:
    raise ValueError(
        "Hugging Face token not found. Add a write token to "
        "Colab Secrets as 'HF_Writing' or define HF_TOKEN."
    )

hf_api = HfApi(token=HF_UPLOAD_TOKEN)

# ------------------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------------------

def write_json_safe(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as file:
        json.dump(
            obj,
            file,
            indent=2,
            ensure_ascii=False,
        )


def get_year_parquet_path(year):
    return (
        CHECKPOINT_DIR_YEARLY
        / f"SatPM_V6GL03_PM25_Bolivia_Municipality_Stats_{year}.parquet"
    )


def get_year_validation_path(year):
    return (
        CHECKPOINT_DIR_YEARLY
        / f"SatPM_V6GL03_PM25_Bolivia_Municipality_Stats_{year}_validation.json"
    )


def validate_year_for_upload(df_year, year):
    expected_rows = int(len(gdf_admin) * 12)

    mean_column = (
        "pm25_mean_valid_cells"
        if "pm25_mean_valid_cells" in df_year.columns
        else "pm25_mean"
    )

    count_column = (
        "pm25_valid_cell_count"
        if "pm25_valid_cell_count" in df_year.columns
        else "pm25_count"
    )

    validation = {
        "year": int(year),
        "rows": int(len(df_year)),
        "expected_rows": expected_rows,
        "unique_municipalities": int(df_year["ID_ADMIN"].nunique()),
        "expected_municipalities": int(len(gdf_admin)),
        "unique_months": int(df_year["month"].nunique()),
        "duplicate_municipality_month_rows": int(
            df_year.duplicated(
                ["ID_ADMIN", "year", "month"]
            ).sum()
        ),
        "missing_pm25_mean_valid_cells": int(
            df_year[mean_column].isna().sum()
        ),
        "zero_valid_cell_count": int(
            df_year[count_column].fillna(0).eq(0).sum()
        ),
        "fallback_rows": int(
            df_year[
                "pm25_used_all_touched_fallback"
            ].fillna(False).sum()
        )
        if "pm25_used_all_touched_fallback" in df_year.columns
        else 0,
        "municipalities_by_month": {
            int(month): int(count)
            for month, count in (
                df_year
                .groupby("month")["ID_ADMIN"]
                .nunique()
                .to_dict()
                .items()
            )
        },
    }

    validation["is_clean"] = (
        validation["rows"] == validation["expected_rows"]
        and validation["unique_municipalities"]
        == validation["expected_municipalities"]
        and validation["unique_months"] == 12
        and validation["duplicate_municipality_month_rows"] == 0
        and validation["missing_pm25_mean_valid_cells"] == 0
        and validation["zero_valid_cell_count"] == 0
    )

    return validation


def upload_year_outputs_to_hf(
    year,
    parquet_path,
    validation_path,
):
    parquet_path = Path(parquet_path)
    validation_path = Path(validation_path)

    if not parquet_path.exists():
        raise FileNotFoundError(
            f"Parquet file not found: {parquet_path}"
        )

    parquet_repo_path = (
        f"{OUTPUT_HF_BASE_FOLDER}/"
        f"SatPM_V6GL03_PM25_Bolivia_Municipality_Stats_{year}.parquet"
    )

    print("Uploading Parquet to:", parquet_repo_path)

    hf_api.upload_file(
        path_or_fileobj=str(parquet_path),
        path_in_repo=parquet_repo_path,
        repo_id=OUTPUT_HF_REPO_ID,
        repo_type=OUTPUT_HF_REPO_TYPE,
        token=HF_UPLOAD_TOKEN,
        commit_message=(
            f"Add municipality monthly PM2.5 panel for {year}"
        ),
    )

    if validation_path.exists():
        validation_repo_path = (
            f"{OUTPUT_HF_BASE_FOLDER}/metadata/"
            f"SatPM_V6GL03_PM25_Bolivia_Municipality_Stats_"
            f"{year}_validation.json"
        )

        print(
            "Uploading validation JSON to:",
            validation_repo_path,
        )

        hf_api.upload_file(
            path_or_fileobj=str(validation_path),
            path_in_repo=validation_repo_path,
            repo_id=OUTPUT_HF_REPO_ID,
            repo_type=OUTPUT_HF_REPO_TYPE,
            token=HF_UPLOAD_TOKEN,
            commit_message=(
                f"Add PM2.5 validation for {year}"
            ),
        )

    print(f"Hugging Face upload completed for {year}.")


def cleanup_temp_geotiffs_for_year(year):
    patterns = [
        f"*{year}*.tif",
        f"*{year}*.tiff",
        f"*{year}*.tif.aux.xml",
        f"*{year}*.tiff.aux.xml",
    ]

    deleted_files = []

    for pattern in patterns:
        for path in TEMP_RASTER_DIR.rglob(pattern):
            if path.is_file():
                try:
                    path.unlink()
                    deleted_files.append(str(path))
                except Exception as error:
                    print(f"Could not delete {path}: {error}")

    print(
        f"Deleted temporary GeoTIFF files for "
        f"{year}: {len(deleted_files)}"
    )

    return deleted_files

# ------------------------------------------------------------------------------
# Production loop
# ------------------------------------------------------------------------------

production_errors = []
production_summaries = []

print("--------------------------------------------------")
print("Full PM2.5 production")
print(f"Years              : {FULL_START_YEAR}–{FULL_END_YEAR}")
print(f"Month workers      : {MONTH_WORKERS}")
print(f"HF repository      : {OUTPUT_HF_REPO_ID}")
print(f"HF destination     : {OUTPUT_HF_BASE_FOLDER}")
print("--------------------------------------------------")

if RUN_PRODUCTION:

    for year in tqdm(
        YEARS_TO_PROCESS,
        desc="Processing years",
    ):
        try:
            print("==================================================")
            print(f"Processing year: {year}")
            print("==================================================")

            df_year = process_year(
                year,
                overwrite=OVERWRITE_YEARLY_CHECKPOINTS,
                month_workers=MONTH_WORKERS,
            )

            parquet_path = get_year_parquet_path(year)
            parquet_path.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            df_year.to_parquet(
                parquet_path,
                index=False,
            )

            print("Saved yearly Parquet:", parquet_path)
            print(
                "Parquet size MB:",
                f"{parquet_path.stat().st_size / (1024 ** 2):.2f}",
            )

            validation = validate_year_for_upload(
                df_year,
                year,
            )

            validation_path = get_year_validation_path(year)

            write_json_safe(
                validation,
                validation_path,
            )

            print(json.dumps(
                validation,
                indent=2,
                ensure_ascii=False,
            ))

            production_summaries.append(validation)

            if validation["is_clean"]:

                if UPLOAD_YEARLY_TO_HF:
                    upload_year_outputs_to_hf(
                        year=year,
                        parquet_path=parquet_path,
                        validation_path=validation_path,
                    )

                if DELETE_TEMP_GEOTIFFS_AFTER_UPLOAD:
                    deleted_files = (
                        cleanup_temp_geotiffs_for_year(year)
                    )

                    validation["deleted_temp_geotiffs"] = (
                        len(deleted_files)
                    )

                    write_json_safe(
                        validation,
                        validation_path,
                    )

            else:
                print(
                    f"Year {year} was not uploaded because "
                    "validation was not clean."
                )

        except Exception as error:
            production_errors.append({
                "year": int(year),
                "error": str(error),
            })

            print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
            print(f"Year {year} failed:")
            print(error)
            print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")

    production_summary_path = (
        METADATA_DIR
        / "production_yearly_summaries.json"
    )

    write_json_safe(
        production_summaries,
        production_summary_path,
    )

    if (
        UPLOAD_YEARLY_TO_HF
        and production_summary_path.exists()
    ):
        hf_api.upload_file(
            path_or_fileobj=str(production_summary_path),
            path_in_repo=(
                f"{OUTPUT_HF_BASE_FOLDER}/metadata/"
                "production_yearly_summaries.json"
            ),
            repo_id=OUTPUT_HF_REPO_ID,
            repo_type=OUTPUT_HF_REPO_TYPE,
            token=HF_UPLOAD_TOKEN,
            commit_message=(
                "Add PM2.5 yearly production summary"
            ),
        )

    if production_errors:
        errors_path = (
            METADATA_DIR
            / "production_errors.json"
        )

        write_json_safe(
            production_errors,
            errors_path,
        )

        if UPLOAD_YEARLY_TO_HF:
            hf_api.upload_file(
                path_or_fileobj=str(errors_path),
                path_in_repo=(
                    f"{OUTPUT_HF_BASE_FOLDER}/metadata/"
                    "production_errors.json"
                ),
                repo_id=OUTPUT_HF_REPO_ID,
                repo_type=OUTPUT_HF_REPO_TYPE,
                token=HF_UPLOAD_TOKEN,
                commit_message=(
                    "Add PM2.5 production errors"
                ),
            )

        display(pd.DataFrame(production_errors))

    else:
        print("Production completed without recorded errors.")

else:
    print(
        "RUN_PRODUCTION is False. "
        "Set it to True when ready."
    )

## Step 15. Compile yearly checkpoints into the final monthly panel

In [ ]:
def list_yearly_checkpoints():
    files = sorted(CHECKPOINT_DIR.glob("SatPM_V6GL03_PM25_Bolivia_Municipality_Stats_*.parquet"))
    # Exclude possible annual/derived files if naming changes later.
    files = [p for p in files if re.search(r"_Stats_\d{4}\.parquet$", p.name)]
    return files


def compile_final_panel(required_years=None):
    if required_years is None:
        required_years = list(range(START_YEAR, END_YEAR + 1))

    checkpoint_files = list_yearly_checkpoints()
    print("Yearly checkpoints found:", len(checkpoint_files))
    for p in checkpoint_files[:5]:
        print(" -", p.name)
    if len(checkpoint_files) > 5:
        print(" ...")
        for p in checkpoint_files[-5:]:
            print(" -", p.name)

    if not checkpoint_files:
        raise FileNotFoundError("No yearly checkpoints found. Run the one-year test or production loop first.")

    panels = []
    for path in tqdm(checkpoint_files, desc="Reading checkpoints"):
        panels.append(pd.read_parquet(path))
    panel = pd.concat(panels, ignore_index=True)
    panel = panel.sort_values(["year", "month", "ID_ADMIN"]).reset_index(drop=True)

    # Restrict to configured period in case the checkpoint folder contains extra years.
    assert START_YEAR == 1998 and END_YEAR == 2024, (
        f"Compilation period was unexpectedly redefined: {START_YEAR}–{END_YEAR}"
    )
    panel = panel[
        (panel["year"] >= START_YEAR)
        & (panel["year"] <= END_YEAR)
    ].copy()

    actual_years = sorted(panel["year"].dropna().astype(int).unique().tolist())
    missing_years = sorted(set(required_years) - set(actual_years))
    if missing_years:
        raise ValueError(f"Missing yearly checkpoints: {missing_years}")

    panel.to_parquet(FINAL_PARQUET, index=False)
    panel.to_csv(FINAL_CSV, index=False)
    shutil.make_archive(str(FINAL_ZIP).replace(".zip", ""), "zip", OUTPUT_DIR, FINAL_CSV.name)

    print("Final panel rows:", len(panel))
    print("Saved parquet:", FINAL_PARQUET)
    print("Saved CSV:", FINAL_CSV)
    print("Saved zipped CSV:", FINAL_ZIP)
    return panel

COMPILE_FINAL_PANEL = False  # Set to True after the relevant checkpoints exist.

if COMPILE_FINAL_PANEL:
    pm25_panel = compile_final_panel()
    display(pm25_panel.head())
else:
    print("COMPILE_FINAL_PANEL is False. Set it to True after processing yearly checkpoints.")

## Step 16. Final validation and publication

Final panel compilation is opt-in and requires all 27 yearly checkpoints. Public publishing is blocked if key counts or required year/month coverage fail.

In [ ]:
def validate_final_panel(panel):
    expected_months = (END_YEAR - START_YEAR + 1) * 12
    expected_rows = len(gdf_admin) * expected_months

    report = {
        "product_name": PRODUCT_NAME,
        "country": "Bolivia",
        "start_year": START_YEAR,
        "end_year": END_YEAR,
        "expected_admin_units": EXPECTED_ADMIN_UNITS,
        "actual_admin_units_in_boundaries": int(len(gdf_admin)),
        "unique_admin_units_in_panel": int(panel["ID_ADMIN"].nunique()),
        "expected_months": int(expected_months),
        "unique_year_months_in_panel": int(panel[["year", "month"]].drop_duplicates().shape[0]),
        "expected_rows": int(expected_rows),
        "actual_rows": int(len(panel)),
        "duplicate_admin_month_rows": int(panel.duplicated(["ID_ADMIN", "year", "month"]).sum()),
        "missing_pm25_mean": int(panel["pm25_mean"].isna().sum()) if "pm25_mean" in panel.columns else None,
        "invalid_percentile_order_rows": int(
            (
                (panel["pm25_p10"] > panel["pm25_p25"])
                | (panel["pm25_p25"] > panel["pm25_median"])
                | (panel["pm25_median"] > panel["pm25_p75"])
                | (panel["pm25_p75"] > panel["pm25_p90"])
            ).sum()
        ) if all(
            c in panel.columns
            for c in ["pm25_p10", "pm25_p25", "pm25_median", "pm25_p75", "pm25_p90"]
        ) else None,
        "created_utc": datetime.now(timezone.utc).isoformat(),
    }

    month_counts = (
        panel.groupby(["year", "month"])["ID_ADMIN"]
        .nunique()
        .reset_index(name="n_admin_units")
        .sort_values(["year", "month"])
    )
    incomplete_months = month_counts[month_counts["n_admin_units"] != len(gdf_admin)].copy()

    report["incomplete_months"] = incomplete_months.to_dict("records")
    return report, month_counts, incomplete_months

RUN_FINAL_VALIDATION = False  # Set to True after pm25_panel exists or FINAL_PARQUET is written.

if RUN_FINAL_VALIDATION:
    if "pm25_panel" not in globals():
        pm25_panel = pd.read_parquet(FINAL_PARQUET)
    validation_report, month_counts, incomplete_months = validate_final_panel(pm25_panel)
    write_json(validation_report, RUN_METADATA_JSON)
    print(json.dumps(validation_report, indent=2, default=str))
    display(month_counts.head())
    display(month_counts.tail())
    if len(incomplete_months):
        print("Incomplete year-months:")
        display(incomplete_months)
    else:
        print("Every year-month has the expected number of municipalities.")
else:
    print("RUN_FINAL_VALIDATION is False.")

## Bolivia adaptations and scientific safeguards

- 339 historical administrative polygons from the GDO-Bolivia `raw/` source, joined one-to-one to its historical lookup by `asdf_id`.
- Canonical ID: six-digit `ine_code` (`ID_ADMIN`); preserved `asdf_id`, `mun_id`, `gid_3`, provenance, geometry-derived area.
- Original SatPM South America raster variable, NetCDF-to-GeoTIFF conversion, zonal statistics, fallback and monthly processing preserved.
- Annual checkpoint publication to public `faviolc/gdo4Bolivia/pm25_monthly/` only after validation.
- **NaN is not zero**. Report missing PM₂.₅ coverage. An incomplete coverage year should be blocked from publication under the inherited strict `is_clean` criterion.
- `GDO_GIT_REF` defaults to `main`: pin a Git SHA before a final archival rerun.
- Do not claim that Colab's in-memory notebook is automatically an executed `.ipynb`; use the notebook publishing step below with an explicitly saved copy.

## Run order

1. Set `HF_Writing` as a private Colab secret with write access to `faviolc/gdo4Bolivia`.
2. Run installation, settings, raster inventory and 339-unit map crosswalk validation.
3. Inspect the 1999 smoke test and the complete one-year checkpoint.
4. Set `RUN_PRODUCTION=True` in Step 14 only after the qualification succeeds.
5. After all years are uploaded, turn on `COMPILE_FINAL_PANEL=True` and `RUN_FINAL_VALIDATION=True`.
6. Run the **final publication** cell to upload the 339 map, crosswalk, consolidated panel and audit reports.
7. In Colab, **File → Save a copy in Drive** with outputs, or use **File → Download → Download .ipynb**. Then run the notebook upload cell pointing to the saved `.ipynb` file. It never pretends to capture unseen execution outputs.

## Step 17. Publish canonical geography and fully validated final panel

Run **after** all years have been processed and `RUN_FINAL_VALIDATION=True` has produced the audit report. Requires a write token. Never publish a partial final panel.

In [ ]:
PUBLISH_FINAL_ARTIFACTS = False  # explicitly enable after successful full production

def publish_checked_file(path, repo_path):
    if not HF_TOKEN:
        raise PermissionError("Missing HF_Writing token")
    if not Path(path).is_file():
        raise FileNotFoundError(path)
    return api.upload_file(path_or_fileobj=str(path), path_in_repo=repo_path,
                           repo_id=OUTPUT_REPO_ID, repo_type="dataset",
                           token=HF_TOKEN, commit_message=f"Publish Bolivia 339 PM2.5: {repo_path}")

if PUBLISH_FINAL_ARTIFACTS:
    if not FINAL_PARQUET.exists() or not RUN_METADATA_JSON.exists():
        raise RuntimeError("Run full panel compilation and final validation first")
    panel_to_publish = pd.read_parquet(FINAL_PARQUET)
    report, counts, incomplete = validate_final_panel(panel_to_publish)
    expected = 339 * 12 * (END_YEAR - START_YEAR + 1)
    if (len(panel_to_publish) != expected or panel_to_publish["ID_ADMIN"].nunique() != 339
        or len(counts) != 12 * (END_YEAR - START_YEAR + 1) or len(incomplete)
        or panel_to_publish.duplicated(["ID_ADMIN", "year", "month"]).any()):
        raise ValueError("Final structural validation failed. Nothing uploaded.")
    files_to_publish = [
        (FINAL_PARQUET, "pm25_monthly/bolivia_pm25_monthly_1998_2024.parquet"),
        (FINAL_CSV, "pm25_monthly/bolivia_pm25_monthly_1998_2024.csv"),
        (RUN_METADATA_JSON, "pm25_monthly/metadata/final_validation.json"),
        (map_path, "geography/bolivia_339_municipalities.geojson"),
        (lookup_path, "geography/municipality_crosswalk_339.csv"),
    ]
    for file_path, remote_path in files_to_publish:
        publish_checked_file(file_path, remote_path)
        print("Published:", remote_path)
else:
    print("Final publication disabled; set PUBLISH_FINAL_ARTIFACTS=True after auditing.")

## Step 18. Archive the **actually executed notebook** to the public dataset

**Colab limitation:** the kernel cannot reliably read the active notebook with all current cell outputs by its own filename. Save/download the executed `.ipynb` via Colab first; mount Drive and give the full path below, or upload the file to `/content`. The publishing cell checks that the notebook has execution counts and output cells. Do not publish notebooks containing credentials or private cell output.

In [ ]:
PUBLISH_EXECUTED_NOTEBOOK = False  # enable only after saving the executed copy
EXECUTED_NOTEBOOK_PATH = ""  # e.g. /content/drive/MyDrive/Bolivia_Monthly_PM25_executed.ipynb

if PUBLISH_EXECUTED_NOTEBOOK:
    import nbformat
    if not EXECUTED_NOTEBOOK_PATH:
        raise ValueError("Set EXECUTED_NOTEBOOK_PATH to an already saved executed .ipynb")
    notebook_file = Path(EXECUTED_NOTEBOOK_PATH)
    if not notebook_file.is_file() or notebook_file.suffix != ".ipynb":
        raise FileNotFoundError(notebook_file)
    executed_nb = nbformat.read(notebook_file, as_version=4)
    executed_cells = sum(c.cell_type == "code" and c.get("execution_count") is not None
                         for c in executed_nb.cells)
    output_cells = sum(c.cell_type == "code" and bool(c.get("outputs"))
                       for c in executed_nb.cells)
    if not executed_cells or not output_cells:
        raise ValueError("Notebook lacks execution counts or output cells; save with outputs first")
    # REVIEW saved notebook output for secrets before enabling publication.
    publish_checked_file(notebook_file, "notebooks/Bolivia_Monthly_PM25_executed.ipynb")
    print("Published executed notebook:", executed_cells, "executed code cells;", output_cells, "cells with outputs")
else:
    print("Save executed .ipynb from Colab and set PUBLISH_EXECUTED_NOTEBOOK=True.")

## Step 19. Publish a clean reproducible notebook source (optional)

The saved source notebook may be uploaded separately even before full execution. This does not replace the **executed** notebook required above.

In [ ]:
PUBLISH_NOTEBOOK_SOURCE = False
SOURCE_NOTEBOOK_PATH = ""  # path to this saved Bolivia_Monthly_PM25_339.ipynb in Colab
if PUBLISH_NOTEBOOK_SOURCE:
    if not SOURCE_NOTEBOOK_PATH or not Path(SOURCE_NOTEBOOK_PATH).is_file():
        raise FileNotFoundError("Provide the saved source notebook in SOURCE_NOTEBOOK_PATH")
    publish_checked_file(SOURCE_NOTEBOOK_PATH, "notebooks/Bolivia_Monthly_PM25_339.ipynb")
    print("Published reproducible notebook source")
else:
    print("Optional source-notebook publication disabled.")